<a href="https://www.kaggle.com/code/shamanthakreddymallu/fertilizer-prediction?scriptVersionId=247236377" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Importing Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from itertools import combinations
import warnings
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

In [ ]:
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
train = pd.read_csv("/kaggle/input/playground-series-s5e6/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e6/test.csv")
original = pd.read_csv("/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv")
submission = pd.read_csv("/kaggle/input/playground-series-s5e6/sample_submission.csv")

# Feature Engineering

In [ ]:
original_copy = original.copy()
for _ in range(6):
    original = pd.concat([original, original_copy], axis=0)

In [ ]:
numerical_features = [
    col for col in train.select_dtypes(include=['int64', 'float64']).columns 
    if col != 'id'
]

In [ ]:
for df in [train, test, original]:
    for col in numerical_features:
        df[f'{col}_Binned'] = df[col].astype(str).astype('category')
    df.rename(columns={'Temparature': 'Temperature'}, inplace=True)
    for col in df.columns:
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int8')
        elif df[col].dtype == 'float64':
            df[col] = df[col].astype('float16')

# Data Pre processing

In [ ]:
cat_cols = [
    col for col in train.select_dtypes(include=['object', 'category']).columns
    if col != "Fertilizer Name"
]

In [ ]:
for col in cat_cols:
    label_enc = LabelEncoder()
    train[col] = label_enc.fit_transform(train[col])
    original[col] = label_enc.transform(original[col])
    test[col] = label_enc.transform(test[col])

In [ ]:
target_enc = LabelEncoder()
train["Fertilizer Name"] = target_enc.fit_transform(train["Fertilizer Name"])
original["Fertilizer Name"] = target_enc.transform(original["Fertilizer Name"])

In [ ]:
for col in cat_cols:
    train[col] = train[col].astype("category")
    original[col] = original[col].astype("category")
    test[col] = test[col].astype("category")

In [ ]:
X = train.drop(columns=["id", "Fertilizer Name"])
y = train["Fertilizer Name"]
X_test = test.drop(columns=["id"])
X_original = original.drop(columns=["Fertilizer Name"])
y_original = original["Fertilizer Name"]

# Predictive Modeling

In [ ]:
def mapk(actual, predicted, k=3):
    def apk(a, p, k):
        p = p[:k]
        score = 0.0
        hits = 0
        seen = set()
        for i, pred in enumerate(p):
            if pred in a and pred not in seen:
                hits += 1
                score += hits / (i + 1.0)
                seen.add(pred)
        return score / min(len(a), k)
    return np.mean([apk(a, p, k) for a, p in zip(actual, predicted)])

In [ ]:
model_configs = {
    'xgb': {
        'model': XGBClassifier,
        'params': {
            'objective': 'multi:softprob',
            'num_class': len(np.unique(y)),
            'max_depth': 8,
            'learning_rate': 0.03,
            'subsample': 0.8,
            'max_bin': 128,
            'colsample_bytree': 0.3,
            'colsample_bylevel': 1,
            'colsample_bynode': 1,
            'tree_method': 'hist',
            'random_state': 42,
            'eval_metric': 'mlogloss',
            'device': 'cuda',
            'enable_categorical': True,
            'n_estimators': 10000,
            'early_stopping_rounds': 50
        }
    },
    'lgb_goss': {
        'model': LGBMClassifier,
        'params': {
            'objective': 'multiclass',
            'num_class': len(np.unique(y)),
            'boosting_type': 'goss',
            'device': 'gpu',
            'colsample_bytree': 0.3275,
            'learning_rate': 0.02670,
            'max_depth': 9,
            'min_child_samples': 84,
            'n_estimators': 10000,
            'n_jobs': -1,
            'num_leaves': 229,
            'random_state': 42,
            'reg_alpha': 6.87997,
            'reg_lambda': 4.7391,
            'subsample': 0.5411,
            'categorical_feature': cat_cols,
            'verbose': -1
        }
    },
    'lgb': {
        'model': LGBMClassifier,
        'params': {
            'objective': 'multiclass',
            'num_class': len(np.unique(y)),
            'device': 'gpu',
            'colsample_bytree': 0.4366,
            'learning_rate': 0.02617,
            'max_depth': 11,
            'min_child_samples': 67,
            'n_estimators': 10000,
            'n_jobs': -1,
            'num_leaves': 243,
            'random_state': 42,
            'reg_alpha': 6.38283,
            'reg_lambda': 9.39295,
            'subsample': 0.79898,
            'categorical_feature': cat_cols,
            'verbose': -1
        }
    }
}

In [ ]:
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)
oof_preds = {name: np.zeros((len(X), y.nunique())) for name in model_configs}
test_preds = {name: np.zeros((len(X_test), y.nunique())) for name in model_configs}
map3_scores = {name: [] for name in model_configs}

In [ ]:
for name, config in model_configs.items():
    print(f"Training {name}...")
    model = config['model'](**config['params'])
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
        x_train, x_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train_fold, y_valid_fold = y.iloc[train_idx], y.iloc[valid_idx]

        x_train = pd.concat([x_train, X_original], axis=0, ignore_index=True)
        y_train_fold = pd.concat([y_train_fold, y_original], axis=0, ignore_index=True)

        if name == 'xgb':
            model.fit(x_train, y_train_fold, eval_set=[(x_train, y_train_fold), (x_valid, y_valid_fold)], verbose=0)
        else:
            model.fit(x_train, y_train_fold, eval_set=[(x_valid, y_valid_fold)], eval_metric='multi_logloss', callbacks=[lgb.early_stopping(stopping_rounds=100)])

        oof_preds[name][valid_idx] = model.predict_proba(x_valid)
        test_preds[name] += model.predict_proba(X_test) / 7

        top_3 = np.argsort(oof_preds[name][valid_idx], axis=1)[:, -3:][:, ::-1]
        score = mapk([[lab] for lab in y_valid_fold], top_3)
        map3_scores[name].append(score)
        print(f"{name} Fold {fold+1}: MAP@3 {score:.5f}")

    print(f"{name} Average MAP@3: {np.mean(map3_scores[name]):.5f}")

In [ ]:
stacking_train = np.hstack([oof_preds[name] for name in oof_preds])
stacking_test = np.hstack([test_preds[name] for name in test_preds])

meta_model = LGBMClassifier(
    objective='multiclass',
    num_class=len(np.unique(y)),
    learning_rate=0.03,
    n_estimators=10000,
    random_state=42,
    verbose=-1
)

In [ ]:
print("Training stacking ensemble...")
final_oof = np.zeros((len(y), len(np.unique(y))))
final_test = np.zeros((len(X_test), len(np.unique(y))))
ensemble_scores = []

In [ ]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(stacking_train, y)):
    x_tr, x_val = stacking_train[train_idx], stacking_train[valid_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[valid_idx]

    meta_model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], eval_metric='multi_logloss', callbacks=[lgb.early_stopping(stopping_rounds=100)])
    final_oof[valid_idx] = meta_model.predict_proba(x_val)
    final_test += meta_model.predict_proba(stacking_test) / 7

    top_3 = np.argsort(final_oof[valid_idx], axis=1)[:, -3:][:, ::-1]
    score = mapk([[lab] for lab in y_val], top_3)
    ensemble_scores.append(score)
    print(f"Ensemble Fold {fold+1}: MAP@3 {score:.5f}")

print(f"Ensemble Average MAP@3: {np.mean(ensemble_scores):.5f}")

# Exporting the Prediction

In [ ]:
output_dir = 'results'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
np.save(f'{output_dir}/stacking_oof.npy', final_oof)
np.save(f'{output_dir}/stacking_test.npy', final_test)
for name in oof_preds:
    np.save(f'{output_dir}/{name}_oof.npy', oof_preds[name])
    np.save(f'{output_dir}/{name}_test.npy', test_preds[name])

In [ ]:
top_3 = np.argsort(final_test, axis=1)[:, -3:][:, ::-1]
labels = target_enc.inverse_transform(top_3.ravel()).reshape(top_3.shape)

In [ ]:
submission = pd.DataFrame({'id': submission['id'],'Fertilizer Name': [' '.join(row) for row in labels]})
submission.to_csv('submission.csv', index=False)

In [ ]:
with open(f'{output_dir}/scores.txt', 'w') as f:
    for name, scores in map3_scores.items():
        f.write(f"{name} MAP@3 Scores: {scores}\n")
        f.write(f"{name} Average MAP@3: {np.mean(scores):.5f}\n")
    f.write(f"Ensemble MAP@3 Scores: {ensemble_scores}\n")
    f.write(f"Ensemble Average MAP@3: {np.mean(ensemble_scores):.5f}\n")